In [ ]:
import pandas as pd
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df = pd.read_csv("/content/water_potability.csv")
df.head()

In [ ]:
# data = data.dropna()
df.isnull().sum()

In [ ]:
# Cleaning data by removing the missing values

df_clean = df.dropna()

In [ ]:
X = df_clean.drop('Potability', axis=1)
y = df_clean['Potability']

from sklearn.model_selection import train_test_split
X_train_clean, X_test_clean, y_train_clean, y_test_clean = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

In [ ]:
# Model Training and Evaluation
models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "SVM": SVC(),
    "KNN": KNeighborsClassifier(),
    "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "XGBoost": XGBClassifier()
}

for name, model in models.items():
    model.fit(X_train_clean, y_train_clean)
    y_pred = model.predict(X_test_clean)
    acc = accuracy_score(y_test_clean, y_pred)
    print(f"Accuracy of {name}: {acc}")


In [ ]:
def train_and_evaluate(X_train, y_train):
  # List of classifiers to evaluate
  classifiers = {
    'Naive Bayes': MultinomialNB(),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Support Vector Classifier': SVC(random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42)
  }

  X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

  for name, model in classifiers.items():
    # Train the classifier
    model.fit(X_train, y_train)

    # Make predictions
    y_pred = model.predict(X_test)

    # Evaluate the classifier
    accuracy = accuracy_score(y_test, y_pred)
    print(f'{name}: {accuracy * 100:.2f}%')

In [ ]:
from sklearn.model_selection import KFold
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

def train_and_evaluate(X, y):
    # List of classifiers to evaluate
    classifiers = {
        'Naive Bayes': MultinomialNB(),
        'Logistic Regression': LogisticRegression(random_state=42, max_iter=10000),
        'Decision Tree': DecisionTreeClassifier(random_state=42),
        'Support Vector Classifier': SVC(random_state=42),
        'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
        'Random Forest': RandomForestClassifier(random_state=42),
        'Gradient Boosting': GradientBoostingClassifier(random_state=42),
        'XGBoost': XGBClassifier(random_state=42)
    }

    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    if isinstance(X, pd.DataFrame):
        X = X.values
    if isinstance(y, pd.Series):
        y = y.values

    for name, model in classifiers.items():
        accuracies = []
        for train_index, test_index in kf.split(X):
            X_train, X_test = X[train_index], X[test_index]
            y_train, y_test = y[train_index], y[test_index]
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            accuracies.append(accuracy_score(y_test, y_pred))
        print(f'{name}: {np.mean(accuracies) * 100:.2f}%')

In [ ]:
train_and_evaluate(X_train_clean, y_train_clean)

**Train Test Split again**

In [ ]:
X = df.drop('Potability', axis=1)
y = df['Potability']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

**Imputation**

In [ ]:
# Imputation is done here
# Imputing into null values
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='mean')
X_train = imputer.fit_transform(X_train)

train_and_evaluate(X_train, y_train)

**Scaling**

In [ ]:
# Scaling is done here

from sklearn.preprocessing import StandardScaler, MinMaxScaler

# scaler = StandardScaler()
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)

train_and_evaluate(X_train_scaled, y_train)

**Imbalance Removal by Adding Synthetic Data**

In [ ]:
y_train.value_counts()

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

train_and_evaluate(X_train_resampled, y_train_resampled)

**Feature Importance Calculation**

In [ ]:
from sklearn.inspection import permutation_importance

rf_classifier = RandomForestClassifier(random_state=42)
rf_classifier.fit(X_train_resampled, y_train_resampled)

# Perform permutation feature importance
result = permutation_importance(rf_classifier, X_train_resampled, y_train_resampled, n_repeats=10, random_state=42)

# Get feature importances
importances = result.importances_mean

# Print feature importances
for i, feature_name in enumerate(X.columns):
    print(f"{feature_name}: {importances[i]}")

In [ ]:
important_features = ['ph', 'Sulfate', 'Solids', 'Hardness', 'Chloramines']

train_and_evaluate(X_train_resampled[:, [X.columns.get_loc(feature) for feature in important_features]], y_train_resampled)

**Adding Meta Features**

In [ ]:
X_train_resampled_meta = pd.DataFrame(X_train_resampled, columns=X.columns)

X_train_resampled_meta['turbidity_organic_interaction'] = (
    X_train_resampled_meta['Turbidity'] * X_train_resampled_meta['Organic_carbon']
)

X_train_resampled_meta['conductivity_turbidity_ratio'] = (
    X_train_resampled_meta['Conductivity'] / (X_train_resampled_meta['Turbidity'] + 1e-6)
)

X_train_resampled_meta['log_Solids'] = np.log1p(X_train_resampled_meta['Solids'])

train_and_evaluate(X_train_resampled_meta, y_train_resampled)

In [ ]:
X_test_scaled=scaler.transform(X_test)
X_test_imputer=imputer.transform(X_test)
X_test_imputer

In [ ]:
from sklearn.ensemble import RandomForestClassifier

final_model = RandomForestClassifier(random_state=42)
final_model.fit(X_train_resampled, y_train_resampled)
y_pred = final_model.predict(X_test_imputer)
accuracy = accuracy_score(y_test, y_pred)
print(accuracy)
